<a href="https://colab.research.google.com/github/anitejpadyana/AI-lab/blob/main/Forward_Chaining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import itertools

# Check whether a symbol is a variable
def is_variable(x):
    return isinstance(x, str) and x[0].islower()


# Unification algorithm
def unify(x, y, theta=None):
    if theta is None:
        theta = {}

    if theta == "fail":
        return "fail"

    elif x == y:
        return theta

    elif is_variable(x):
        return unify_var(x, y, theta)

    elif is_variable(y):
        return unify_var(y, x, theta)

    elif isinstance(x, tuple) and isinstance(y, tuple) and len(x) == len(y):
        if x[0] != y[0]:
            return "fail"

        for a, b in zip(x[1:], y[1:]):
            theta = unify(a, b, theta)

            if theta == "fail":
                return "fail"

        return theta

    else:
        return "fail"


# Unify variables
def unify_var(var, x, theta):
    if var in theta:
        return unify(theta[var], x, theta)

    elif x in theta:
        return unify(var, theta[x], theta)

    else:
        new_theta = theta.copy()
        new_theta[var] = x
        return new_theta


# Apply substitution
def subst(theta, expr):
    if isinstance(expr, tuple):
        return tuple(subst(theta, e) for e in expr)

    elif is_variable(expr) and expr in theta:
        return subst(theta, theta[expr])

    else:
        return expr


# Standardize variables in rules
def standardize_variables(rule, counter):
    premises, conclusion = rule
    mapping = {}

    def replace(expr):
        if isinstance(expr, tuple):
            return tuple(replace(e) for e in expr)

        elif is_variable(expr):
            if expr not in mapping:
                mapping[expr] = f"{expr}_{next(counter)}"

            return mapping[expr]

        return expr

    new_premises = [replace(p) for p in premises]
    new_conclusion = replace(conclusion)

    return new_premises, new_conclusion


# Format an atom
def format_atom(atom):
    return f"{atom[0]}({', '.join(atom[1:])})"


# Format a rule
def format_rule(rule):
    premises, conclusion = rule

    prem_str = " ∧ ".join(format_atom(p) for p in premises)

    return f"{prem_str} ⇒ {format_atom(conclusion)}"


# Forward chaining algorithm
def fol_fc_ask(facts, rules, alpha):
    kb_facts = set(facts)
    counter = itertools.count(1)
    step = 1

    while True:
        new_facts = set()

        print(f"Iteration - {step}")

        for rule in rules:
            premises, conclusion = standardize_variables(rule, counter)

            def find_substitutions(prem_list, current_theta):
                if not prem_list:
                    yield current_theta
                    return

                first_prem = subst(current_theta, prem_list[0])

                for fact in kb_facts:
                    theta = unify(first_prem, fact, current_theta)

                    if theta != "fail":
                        yield from find_substitutions(
                            prem_list[1:], theta
                        )

            for theta in find_substitutions(premises, {}):
                q_prime = subst(theta, conclusion)

                if q_prime not in kb_facts and q_prime not in new_facts:
                    new_facts.add(q_prime)

                    print(f"Inferred: {format_atom(q_prime)}")

                    phi = unify(q_prime, alpha, {})

                    if phi != "fail":
                        kb_facts.update(new_facts)
                        return True, phi

        if not new_facts:
            return False, None

        kb_facts.update(new_facts)
        step += 1


# Given problem statements
statements = [
    "As per the law, it is a crime for an American to sell weapons to hostile nations.",
    "Country A, an enemy of America, has some missiles.",
    "All the missiles were sold to it by Robert, who is an American citizen."
]


# Knowledge Base - Facts
facts = [
    ("American", "Robert"),
    ("Enemy", "A", "America"),
    ("Owns", "A", "T1"),
    ("Missile", "T1")
]


# Knowledge Base - Rules
rules = [
    (
        [
            ("American", "p"),
            ("Weapon", "q"),
            ("Sells", "p", "q", "r"),
            ("Hostile", "r")
        ],
        ("Criminal", "p")
    ),

    (
        [
            ("Missile", "x"),
            ("Owns", "A", "x")
        ],
        ("Sells", "Robert", "x", "A")
    ),

    (
        [("Missile", "x")],
        ("Weapon", "x")
    ),

    (
        [("Enemy", "x", "America")],
        ("Hostile", "x")
    )
]


# Query
query = ("Criminal", "Robert")


# Display problem statements
print("GIVEN PROBLEM STATEMENTS")

for s in statements:
    print(f"- {s}")


# Display knowledge base
print("\nKNOWLEDGE BASE (FOL REPRESENTATION)")

print("Facts:")
for f in facts:
    print(f"  {format_atom(f)}")


# Display rules
print("\nRules:")

for r in rules:
    print(f"  {format_rule(r)}")


# Display query
print(f"\nTO PROVE\n  {format_atom(query)}\n")


# Execute forward chaining
print("FORWARD CHAINING PROOF")

proved, substitution = fol_fc_ask(facts, rules, query)


# Display final result
print("\nRESULT")

if proved:
    print(
        f"Query {format_atom(query)} is PROVED! "
        f"(Substitution: {substitution})"
    )
else:
    print(f"Query {format_atom(query)} CANNOT be proved.")


GIVEN PROBLEM STATEMENTS
- As per the law, it is a crime for an American to sell weapons to hostile nations.
- Country A, an enemy of America, has some missiles.
- All the missiles were sold to it by Robert, who is an American citizen.

KNOWLEDGE BASE (FOL REPRESENTATION)
Facts:
  American(Robert)
  Enemy(A, America)
  Owns(A, T1)
  Missile(T1)

Rules:
  American(p) ∧ Weapon(q) ∧ Sells(p, q, r) ∧ Hostile(r) ⇒ Criminal(p)
  Missile(x) ∧ Owns(A, x) ⇒ Sells(Robert, x, A)
  Missile(x) ⇒ Weapon(x)
  Enemy(x, America) ⇒ Hostile(x)

TO PROVE
  Criminal(Robert)

FORWARD CHAINING PROOF
Iteration - 1
Inferred: Sells(Robert, T1, A)
Inferred: Weapon(T1)
Inferred: Hostile(A)
Iteration - 2
Inferred: Criminal(Robert)

RESULT
Query Criminal(Robert) is PROVED! (Substitution: {})
